In [ ]:
# =========================
# IMPORTS
# =========================
import numpy as np
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping

# =========================
# PARAMETERS
# =========================
vocab_size = 10000
max_len = 150
embedding_dim = 128

# =========================
# LOAD DATA
# =========================
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=vocab_size)

# =========================
# PAD SEQUENCES
# =========================
x_train = pad_sequences(x_train, maxlen=max_len)
x_test = pad_sequences(x_test, maxlen=max_len)

# =========================
# MODEL
# =========================
model = Sequential([
    Embedding(vocab_size, embedding_dim),

    Bidirectional(LSTM(
        64,
        return_sequences=False,
        dropout=0.3,
        recurrent_dropout=0.3
    )),

    Dropout(0.5),

    Dense(1, activation='sigmoid')
])

# =========================
# COMPILE
# =========================
model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

# =========================
# EARLY STOPPING
# =========================
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

# =========================
# TRAIN
# =========================
history = model.fit(
    x_train, y_train,
    epochs=10,
    batch_size=64,
    validation_split=0.2,
    callbacks=[early_stop]
)

# =========================
# EVALUATE
# =========================
loss, acc = model.evaluate(x_test, y_test)
print("Test Accuracy:", acc)

# =========================
# PREDICTION FUNCTION
# =========================
def predict_review(text_sequence):
    padded = pad_sequences([text_sequence], maxlen=max_len)
    prediction = model.predict(padded)[0][0]

    if prediction > 0.5:
        return f"Positive ({prediction:.2f})"
    else:
        return f"Negative ({prediction:.2f})"

# Example (random test sample)
print(predict_review(x_test[0]))

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
Epoch 1/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 308s 946ms/step - accuracy: 0.7370 - loss: 0.5197 - val_accuracy: 0.8026 - val_loss: 0.4568
Epoch 2/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 294s 941ms/step - accuracy: 0.8554 - loss: 0.3470 - val_accuracy: 0.8518 - val_loss: 0.3455
Epoch 3/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 294s 941ms/step - accuracy: 0.8549 - loss: 0.3471 - val_accuracy: 0.8410 - val_loss: 0.3697
Epoch 4/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 319s 931ms/step - accuracy: 0.8969 - loss: 0.2604 - val_accuracy: 0.8418 - val_loss: 0.3772
782/782 ━━━━━━━━━━━━━━━━━━━━ 143s 182ms/step - accuracy: 0.8564 - loss: 0.3423
Test Accuracy: 0.8563600182533264
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 780ms/step
Negative (0.10)


In [ ]:
# =========================
# IMPORTS
# =========================
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense
import numpy as np

# =========================
# PARAMETERS
# =========================
num_samples = 10000
max_len = 5
vocab_size = 50
latent_dim = 128

# =========================
# DATA GENERATION
# =========================
def generate_data(num_samples):
    encoder_input = []
    decoder_input = []
    decoder_output = []

    for _ in range(num_samples):
        seq = np.random.randint(1, vocab_size, size=max_len)
        rev_seq = seq[::-1]

        encoder_input.append(seq)

        # decoder input starts with 0 (start token)
        decoder_input.append(np.insert(rev_seq[:-1], 0, 0))
        decoder_output.append(rev_seq)

    return np.array(encoder_input), np.array(decoder_input), np.array(decoder_output)

encoder_input_data, decoder_input_data, decoder_output_data = generate_data(num_samples)

# =========================
# ONE HOT ENCODING
# =========================
encoder_input_data = np.eye(vocab_size)[encoder_input_data]
decoder_input_data = np.eye(vocab_size)[decoder_input_data]
decoder_output_data = np.eye(vocab_size)[decoder_output_data]

# =========================
# ENCODER
# =========================
encoder_inputs = Input(shape=(None, vocab_size))
encoder_lstm = LSTM(latent_dim, return_state=True)
encoder_outputs, state_h, state_c = encoder_lstm(encoder_inputs)

encoder_states = [state_h, state_c]

# =========================
# DECODER
# =========================
decoder_inputs = Input(shape=(None, vocab_size))
decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)

decoder_outputs, _, _ = decoder_lstm(decoder_inputs, initial_state=encoder_states)

decoder_dense = Dense(vocab_size, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)

# =========================
# MODEL
# =========================
seq2seq_model = Model([encoder_inputs, decoder_inputs], decoder_outputs)

seq2seq_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# =========================
# TRAIN
# =========================
seq2seq_model.fit(
    [encoder_input_data, decoder_input_data],
    decoder_output_data,
    batch_size=64,
    epochs=10,
    validation_split=0.2
)

Epoch 1/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 7s 11ms/step - accuracy: 0.1606 - loss: 3.2286 - val_accuracy: 0.2569 - val_loss: 2.3435
Epoch 2/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.3079 - loss: 2.0417 - val_accuracy: 0.3683 - val_loss: 1.7927
Epoch 3/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.5052 - loss: 1.4832 - val_accuracy: 0.6463 - val_loss: 1.1988
Epoch 4/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.7913 - loss: 0.8857 - val_accuracy: 0.8831 - val_loss: 0.6582
Epoch 5/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9381 - loss: 0.4651 - val_accuracy: 0.9518 - val_loss: 0.3718
Epoch 6/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9785 - loss: 0.2606 - val_accuracy: 0.9774 - val_loss: 0.2245
Epoch 7/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9904 - loss: 0.1647 - val_accuracy: 0.9889 - val_loss: 0.1491
Epoch 8/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9956 - loss: 0.1135 - val_accuracy: 0